# Workshop Lite: AgentCore + LangGraph ReAct (Google Docs)

Спростив флоу для демо:
- мінімум інфра-коду;
- офіційні патерни LangGraph + AgentCore;
- CLI для runtime configure/deploy/invoke.


## Prerequisites

У `.env` мають бути:
- `GOOGLE_CLIENT_ID`
- `GOOGLE_CLIENT_SECRET`
- `GOOGLE_DOC_ID`
- `GOOGLE_API_KEY`

Опціонально:
- `AWS_PROFILE` (default: `workshop`)
- `AWS_REGION` (default: `us-east-1`)
- `WORKSHOP_PREFIX` (default: `acwslite`)
- `OAUTH_RETURN_URL` (default: `http://localhost:8081/oauth2/callback`)


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>📦 Що відбувається у цій клітинці:</strong><br>
  Тут ми імпортуємо стандартні модулі Python, `boto3` для роботи з AWS та `requests` для виконання HTTP-запитів.
</div>


In [ ]:
from __future__ import annotations

import ast
import json
import os
import re
import shlex
import subprocess
import time
import urllib.parse
from pathlib import Path
from typing import Any

import boto3
import requests
from botocore.exceptions import ClientError


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🛠 Що відбувається у цій клітинці:</strong><br>
  Завантажуємо змінні оточення з файлу `.env` (щоб безпечно працювати зі ключами), налаштовуємо середовище та створюємо зручну обгортку `run()` для форматованого виконання термінальних команд команди.
</div>


In [ ]:
ROOT = Path.cwd()
AGENTCORE = str(ROOT / '.venv/bin/agentcore')
ENV_FILE = ROOT / '.env'


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def cli_env() -> dict[str, str]:
    env = os.environ.copy()
    env['AWS_PROFILE'] = env.get('AWS_PROFILE', 'workshop')
    env['AWS_REGION'] = env.get('AWS_REGION', 'us-east-1')
    env['AWS_DEFAULT_REGION'] = env['AWS_REGION']
    for k in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN'):
        env.pop(k, None)
    return env


def run(cmd: str, check: bool = True) -> str:
    print(f"\n🚀 Виконую команду: {cmd}")
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=cli_env(),
        bufsize=1,
    )

    lines: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    run.last_rc = rc
    if check and rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")
    if not check and rc != 0:
        print(f"⚠️ [Не критично] Команда завершилась з кодом {rc}")
    return out



run.last_rc = 0



<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>☁️ Що відбувається у цій клітинці:</strong><br>
  Ініціалізуємо підключення до AWS (`boto3.Session`). Визначаємо `Account ID` та генеруємо стандартизовані імена для всіх сервісів та ресурсів (наприклад, ім'я для Gateway, User Pool тощо), використовуючи префікс.
</div>


In [ ]:
load_env_file(ENV_FILE)

required = ['GOOGLE_CLIENT_ID', 'GOOGLE_CLIENT_SECRET', 'GOOGLE_DOC_ID', 'GOOGLE_API_KEY']
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(f"Missing required env vars: {missing}")

AWS_PROFILE = os.getenv('AWS_PROFILE', 'workshop')
AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')
WORKSHOP_PREFIX = os.getenv('WORKSHOP_PREFIX', 'acwslite')
OAUTH_RETURN_URL = os.getenv('OAUTH_RETURN_URL', 'http://localhost:8081/oauth2/callback')
GOOGLE_MODEL_ID = os.getenv('GOOGLE_MODEL_ID', 'gemini-2.5-flash')

# Force subprocess/CLI calls to use the same AWS identity as boto3 in this notebook.
os.environ['AWS_PROFILE'] = AWS_PROFILE
os.environ['AWS_REGION'] = AWS_REGION
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

# Remove stale static creds if they exist (they override profile and often cause InvalidClientTokenId).
for k in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN'):
    if os.getenv(k):
        os.environ.pop(k, None)

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sts = session.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']

cognito = session.client('cognito-idp')
ac_control = session.client('bedrock-agentcore-control')

base = re.sub(r'[^a-zA-Z0-9_]', '_', WORKSHOP_PREFIX)
base = re.sub(r'_+', '_', base).strip('_') or 'acwslite'
dns = re.sub(r'[^a-z0-9-]', '-', base.lower().replace('_', '-')).strip('-') or 'acwslite'

names = {
    'user_pool_name': f'{base}_runtime_pool',
    'user_client_name': f'{base}_runtime_user_client',
    'provider_name': f'{base}_google_provider',
    'gateway_name': f'{dns}-gateway',
    'target_name': f'{dns}-google-docs-target',
    'runtime_agent_name': f'{base}_runtime_agent',
}

state: dict[str, Any] = {'names': names, 'aws_region': AWS_REGION, 'account_id': ACCOUNT_ID}

print('✅ AWS Account ID:', ACCOUNT_ID)
print('👤 AWS Profile:', AWS_PROFILE)
print('🌍 AWS Region:', AWS_REGION)
print('📝 Згенеровані імена ресурсів:\n', json.dumps(names, indent=2))


print('🔁 OAuth return URL:', OAUTH_RETURN_URL)


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔍 Що відбувається у цій клітинці:</strong><br>
  Робимо простий перевірочний (`preflight`) виклик CLI `aws sts get-caller-identity`, щоб переконатися, що у нас коректно налаштовані права доступу і команда в терміналі успішно бачить наш AWS-акаунт.
</div>


In [ ]:
# CLI preflight (must pass before gateway/runtime CLI calls)
_ = run('aws sts get-caller-identity')



## Step 1 - Inbound auth (Cognito)

Створюємо/перевикористовуємо:
- User Pool
- App Client (USER_PASSWORD_AUTH)
- demo user

Отримуємо `access_token` для inbound JWT викликів.


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔑 Що відбувається у цій клітинці:</strong><br>
  Створюємо (або знаходимо існуючий) **Когніто User Pool** та клієнтський додаток. Створюємо тестового користувача для демо та одразу отримуємо `access_token` — з його допомогою ми будемо авторизувати виклики до агента.
</div>


In [ ]:
def find_pool_id_by_name(pool_name: str) -> str | None:
    resp = cognito.list_user_pools(MaxResults=60)
    for item in resp.get('UserPools', []):
        if item.get('Name') == pool_name:
            return item.get('Id')
    return None

pool_id = find_pool_id_by_name(names['user_pool_name'])
if not pool_id:
    created = cognito.create_user_pool(
        PoolName=names['user_pool_name'],
        Policies={
            'PasswordPolicy': {
                'MinimumLength': 10,
                'RequireUppercase': True,
                'RequireLowercase': True,
                'RequireNumbers': True,
                'RequireSymbols': False,
                'TemporaryPasswordValidityDays': 7,
            }
        },
        AutoVerifiedAttributes=['email'],
    )
    pool_id = created['UserPool']['Id']

clients = cognito.list_user_pool_clients(UserPoolId=pool_id, MaxResults=60).get('UserPoolClients', [])
client = next((c for c in clients if c.get('ClientName') == names['user_client_name']), None)
if not client:
    client = cognito.create_user_pool_client(
        UserPoolId=pool_id,
        ClientName=names['user_client_name'],
        GenerateSecret=False,
        ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH', 'ALLOW_USER_SRP_AUTH'],
    )['UserPoolClient']

user_client_id = client['ClientId']
demo_username = os.getenv('DEMO_USERNAME', 'workshop.user')
demo_password = os.getenv('DEMO_PASSWORD', 'DemoPassw0rd2026!')

try:
    cognito.admin_get_user(UserPoolId=pool_id, Username=demo_username)
except cognito.exceptions.UserNotFoundException:
    cognito.admin_create_user(
        UserPoolId=pool_id,
        Username=demo_username,
        TemporaryPassword=demo_password,
        MessageAction='SUPPRESS',
        UserAttributes=[{'Name': 'email', 'Value': 'workshop.user@example.com'}],
    )

cognito.admin_set_user_password(UserPoolId=pool_id, Username=demo_username, Password=demo_password, Permanent=True)

discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{pool_id}/.well-known/openid-configuration"


def get_user_access_token() -> str:
    resp = cognito.initiate_auth(
        ClientId=user_client_id,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': demo_username, 'PASSWORD': demo_password},
    )
    return resp['AuthenticationResult']['AccessToken']


user_access_token = get_user_access_token()
state['inbound'] = {
    'user_pool_id': pool_id,
    'user_client_id': user_client_id,
    'discovery_url': discovery_url,
    'demo_username': demo_username,
}

print('✅ Cognito налаштовано успішно:\n', json.dumps(state['inbound'], indent=2))
print('🎫 Довжина отриманого Access Token:', len(user_access_token))


## Step 2 - Outbound provider + Gateway

1. Створюємо OAuth provider (Google) в AgentCore Identity.
2. Створюємо Gateway з inbound JWT (Cognito discovery + allowed client).
3. Додаємо target з Google Docs OpenAPI + OAuth credential provider.


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔗 Що відбувається у цій клітинці:</strong><br>
  Реєструємо **Google OAuth-провайдер** в сервісі AgentCore (використовуючи `Client ID` та `Secret` з `.env`). Це необхідно, щоб AgentCore міг згенерувати посилання для логіну і авторизуватись у Google від імені нашого юзера.
</div>


In [ ]:
def list_oauth_providers() -> list[dict[str, Any]]:
    items: list[dict[str, Any]] = []
    next_token = None
    while True:
        req = {'maxResults': 20}
        if next_token:
            req['nextToken'] = next_token
        resp = ac_control.list_oauth2_credential_providers(**req)
        items.extend(resp.get('credentialProviders', []))
        next_token = resp.get('nextToken')
        if not next_token:
            break
    return items

provider_exists = any(p.get('name') == names['provider_name'] for p in list_oauth_providers())
provider_req = {
    'name': names['provider_name'],
    'credentialProviderVendor': 'GoogleOauth2',
    'oauth2ProviderConfigInput': {
        'googleOauth2ProviderConfig': {
            'clientId': os.environ['GOOGLE_CLIENT_ID'],
            'clientSecret': os.environ['GOOGLE_CLIENT_SECRET'],
        }
    },
}

if provider_exists:
    provider_op = ac_control.update_oauth2_credential_provider(**provider_req)
else:
    provider_op = ac_control.create_oauth2_credential_provider(**provider_req)

provider_obj = ac_control.get_oauth2_credential_provider(name=names['provider_name'])
provider_arn = provider_obj['credentialProviderArn']
callback_url = provider_op.get('callbackUrl') or provider_obj.get('callbackUrl', '')

state['provider'] = {
    'provider_arn': provider_arn,
    'provider_name': names['provider_name'],
    'callback_url': callback_url,
}

print('✅ OAuth Провайдер налаштовано:\n', json.dumps(state['provider'], indent=2))
print('⚠️ ВАЖЛИВО: callback_url must be in Google OAuth client Authorized redirect URIs (one-time unless callback changes).')


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🚪 Що відбувається у цій клітинці:</strong><br>
  Створюємо **AgentCore Gateway**. Цей Gateway перевіряє наш JWT токен (зроблений у Cognito) і виступає "вхідними дверима" до наших інструментів (tools).
</div>


In [ ]:
authorizer_config = {
    'customJWTAuthorizer': {
        'discoveryUrl': state['inbound']['discovery_url'],
        'allowedClients': [state['inbound']['user_client_id']],
    }
}

DESIRED_MCP_VERSION = '2025-11-25'
iam = session.client('iam')


def list_gateways() -> list[dict[str, Any]]:
    items: list[dict[str, Any]] = []
    next_token = None
    while True:
        req = {'maxResults': 100}
        if next_token:
            req['nextToken'] = next_token
        resp = ac_control.list_gateways(**req)
        items.extend(resp.get('items', []))
        next_token = resp.get('nextToken')
        if not next_token:
            break
    return items


def gateway_has_version(gateway_obj: dict[str, Any], version: str) -> bool:
    versions = ((gateway_obj.get('protocolConfiguration') or {}).get('mcp') or {}).get('supportedVersions') or []
    return version in versions


def ensure_gateway_role() -> str:
    role_name = f"{names['gateway_name']}-role"[:64]
    trust_policy = {
        'Version': '2012-10-17',
        'Statement': [
            {
                'Effect': 'Allow',
                'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
                'Action': 'sts:AssumeRole',
            }
        ],
    }
    inline_policy = {
        'Version': '2012-10-17',
        'Statement': [
            {
                'Effect': 'Allow',
                'Action': [
                    'lambda:InvokeFunction',
                    'execute-api:Invoke',
                    'logs:CreateLogGroup',
                    'logs:CreateLogStream',
                    'logs:PutLogEvents',
                    'xray:PutTraceSegments',
                    'xray:PutTelemetryRecords',
                ],
                'Resource': '*',
            }
        ],
    }

    try:
        role_arn = iam.get_role(RoleName=role_name)['Role']['Arn']
    except iam.exceptions.NoSuchEntityException:
        created = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='AgentCore Workshop Gateway execution role',
        )
        role_arn = created['Role']['Arn']

    iam.put_role_policy(
        RoleName=role_name,
        PolicyName='AgentCoreWorkshopGatewayInlinePolicy',
        PolicyDocument=json.dumps(inline_policy),
    )

    time.sleep(5)
    return role_arn


all_gateways = [g for g in list_gateways() if (g.get('name') or '') == names['gateway_name']]
selected = None
for g in sorted(all_gateways, key=lambda x: str(x.get('updatedAt', '')), reverse=True):
    full = ac_control.get_gateway(gatewayIdentifier=g['gatewayId'])
    if full.get('status') == 'READY' and gateway_has_version(full, DESIRED_MCP_VERSION):
        selected = full
        break

if not selected:
    role_arn = ensure_gateway_role()
    created = ac_control.create_gateway(
        name=names['gateway_name'],
        roleArn=role_arn,
        protocolType='MCP',
        protocolConfiguration={'mcp': {'searchType': 'SEMANTIC', 'supportedVersions': [DESIRED_MCP_VERSION]}},
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration=authorizer_config,
    )

    new_gateway_id = created['gatewayId']
    for _ in range(60):
        full = ac_control.get_gateway(gatewayIdentifier=new_gateway_id)
        if full.get('status') == 'READY':
            selected = full
            break
        time.sleep(3)
    else:
        raise TimeoutError('New gateway did not reach READY')

if not selected:
    raise RuntimeError('No suitable gateway found/created.')

state['gateway'] = {
    'gateway_id': selected['gatewayId'],
    'gateway_arn': selected['gatewayArn'],
    'gateway_url': selected['gatewayUrl'],
    'gateway_role_arn': selected['roleArn'],
    'mcp_supported_versions': ((selected.get('protocolConfiguration') or {}).get('mcp') or {}).get('supportedVersions', []),
    'workload_identity_arn': ((selected.get('workloadIdentityDetails') or {}).get('workloadIdentityArn') or ''),
}

print('Gateway ready:', json.dumps(state['gateway'], indent=2))


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🎯 Що відбувається у цій клітинці:</strong><br>
  Підключаємо **Google Docs Target**. Для цього ми визначаємо мінімальну OpenAPI схему (тільки для `getDocument`) і прив'язуємо її до нашого Gateway. Тепер шлюз знає про інструмент Google Docs.
</div>


In [ ]:
google_docs_openapi = {
    'openapi': '3.0.1',
    'info': {'title': 'Google Docs Minimal API', 'version': '1.0.0'},
    'servers': [{'url': 'https://docs.googleapis.com'}],
    'paths': {
        '/v1/documents/{documentId}': {
            'get': {
                'operationId': 'getDocument',
                'summary': 'Get Google Doc by ID',
                'parameters': [
                    {'name': 'documentId', 'in': 'path', 'required': True, 'schema': {'type': 'string'}}
                ],
                'responses': {'200': {'description': 'Google Docs document payload'}},
            }
        }
    },
}


def list_targets(gateway_id: str) -> list[dict[str, Any]]:
    items: list[dict[str, Any]] = []
    next_token = None
    while True:
        req = {'gatewayIdentifier': gateway_id, 'maxResults': 100}
        if next_token:
            req['nextToken'] = next_token
        resp = ac_control.list_gateway_targets(**req)
        items.extend(resp.get('items', []))
        next_token = resp.get('nextToken')
        if not next_token:
            break
    return items


def ensure_target(
    gateway_id: str,
    target_name: str,
    inline_openapi: dict[str, Any],
    credential_cfg: list[dict[str, Any]],
) -> dict[str, Any]:
    target_cfg = {'mcp': {'openApiSchema': {'inlinePayload': json.dumps(inline_openapi)}}}
    existing = next((t for t in list_targets(gateway_id) if t.get('name') == target_name), None)

    if existing:
        target_id = existing['targetId']
        ac_control.update_gateway_target(
            gatewayIdentifier=gateway_id,
            targetId=target_id,
            name=target_name,
            targetConfiguration=target_cfg,
            credentialProviderConfigurations=credential_cfg,
        )
    else:
        created = ac_control.create_gateway_target(
            gatewayIdentifier=gateway_id,
            name=target_name,
            targetConfiguration=target_cfg,
            credentialProviderConfigurations=credential_cfg,
        )
        target_id = created['targetId']

    for _ in range(40):
        tgt = ac_control.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        if tgt.get('status') == 'READY':
            return tgt
        time.sleep(3)
    raise TimeoutError(f'Target {target_name} did not reach READY')


gateway_id = state['gateway']['gateway_id']


def ensure_workload_identity_return_url(workload_identity_arn: str, return_url: str) -> dict[str, Any] | None:
    if not workload_identity_arn or not return_url:
        return None

    workload_name = workload_identity_arn.rsplit('/', 1)[-1]
    current = ac_control.get_workload_identity(name=workload_name)
    urls = list(current.get('allowedResourceOauth2ReturnUrls') or [])

    if return_url in urls:
        return current

    urls.append(return_url)
    updated = ac_control.update_workload_identity(
        name=workload_name,
        allowedResourceOauth2ReturnUrls=sorted(set(urls)),
    )
    return updated


google_docs_cred_cfg = [{
    'credentialProviderType': 'OAUTH',
    'credentialProvider': {
        'oauthCredentialProvider': {
            'providerArn': state['provider']['provider_arn'],
            'scopes': ['https://www.googleapis.com/auth/documents.readonly'],
            'grantType': 'AUTHORIZATION_CODE',
            'defaultReturnUrl': OAUTH_RETURN_URL,
        }
    },
}]

google_tgt = ensure_target(
    gateway_id=gateway_id,
    target_name=names['target_name'],
    inline_openapi=google_docs_openapi,
    credential_cfg=google_docs_cred_cfg,
)

state['gateway']['target_id'] = google_tgt['targetId']
state['gateway']['google_docs_tool_name'] = f"{names['target_name']}___getDocument"
state['gateway']['default_return_url'] = OAUTH_RETURN_URL
state['gateway']['mcp_version'] = '2025-11-25'

wid_obj = ensure_workload_identity_return_url(
    state['gateway'].get('workload_identity_arn', ''),
    OAUTH_RETURN_URL,
)
if wid_obj is not None:
    state['gateway']['workload_identity_name'] = wid_obj.get('name', '')
    state['gateway']['allowed_return_urls'] = wid_obj.get('allowedResourceOauth2ReturnUrls', [])

print('✅ Target (Google Docs) connected:', json.dumps(state['gateway'], indent=2))
print('🛠 Google Docs tool:', state['gateway']['google_docs_tool_name'])


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🩺 Що відбувається у цій клітинці:</strong><br>
  Перевіряємо, чи всі попередні етапи (Inbound Auth, Outbound Provider, Gateway, Target) ініціалізовані і зв'язані без помилок.
</div>


In [ ]:
# Health gate before runtime deploy
errors = []
if 'gateway' not in state:
    errors.append('state.gateway missing')
else:
    versions = state['gateway'].get('mcp_supported_versions', [])
    if '2025-11-25' not in versions:
        errors.append(f"Gateway MCP version mismatch: {versions}")
    if not state['gateway'].get('google_docs_tool_name'):
        errors.append('google_docs_tool_name missing')

if not state.get('provider', {}).get('callback_url'):
    errors.append('provider.callback_url missing')

if errors:
    raise RuntimeError('Health gate failed: ' + '; '.join(errors))

print('✅ Health gate пройдено успішно: gateway/protocol/provider are aligned for 3LO runtime flow.')
print('💡 Нагадування: callback_url must be present in Google OAuth client redirect URIs.')

# Gateway smoke-test: tools/call should return either
# 1) OAuth elicitation challenge (-32042), or
# 2) successful non-error result.
smoke_headers = {
    'Authorization': f'Bearer {user_access_token}',
    'Content-Type': 'application/json',
    'MCP-Protocol-Version': state['gateway'].get('mcp_version', '2025-11-25'),
}
smoke_body = {
    'jsonrpc': '2.0',
    'id': 1,
    'method': 'tools/call',
    'params': {
        'name': state['gateway']['google_docs_tool_name'],
        'arguments': {'documentId': os.environ['GOOGLE_DOC_ID']},
    },
}
smoke_resp = requests.post(state['gateway']['gateway_url'], headers=smoke_headers, json=smoke_body, timeout=(10, 60))
if smoke_resp.status_code != 200:
    raise RuntimeError(f'Gateway smoke-test HTTP failed: {smoke_resp.status_code} {smoke_resp.text[:300]}')

smoke_json = smoke_resp.json()
smoke_error = smoke_json.get('error')
smoke_result = smoke_json.get('result', {})
smoke_is_error = bool(smoke_result.get('isError'))
smoke_text = ''
for block in smoke_result.get('content', []) or []:
    if isinstance(block, dict) and block.get('type') == 'text':
        smoke_text += str(block.get('text', ''))

if isinstance(smoke_error, dict) and smoke_error.get('code') == -32042:
    elic = ((smoke_error.get('data') or {}).get('elicitations') or [{}])[0]
    print('✅ Gateway smoke-test: OAuth consent challenge is working.')
    print('authorization_url:', elic.get('url', ''))
elif smoke_is_error and 'internal error' in smoke_text.lower():
    raise RuntimeError(
        'Gateway smoke-test failed: Google Docs tool returned generic internal error.\n'
        'Re-run Step 2 (provider + gateway + target) and ensure you use this updated notebook with .venv kernel.'
    )
else:
    print('✅ Gateway smoke-test: tool call path is healthy.')


## Step 3 - Local ReAct Agent (Mock Data)

Цей крок навмисно ізольований від AWS/Gateway OAuth, щоб локальне навчання було стабільним.

Що тут є:
- `ChatGoogleGenerativeAI` як LLM
- `@tool` + `create_react_agent`
- Один мок-інструмент: `get_workshop_doc`
- Локальний invoke без зовнішнього OAuth


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🤖 Що відбувається у цій клітинці:</strong><br>
  Створюємо локального агента (`LangGraph ReAct`). Визначаємо захардкоджені (**мок**) версії інструментів, щоб можна було протестувати логіку розмірковування агента локально та швидко, без залучення зовнішніх API та інфраструктури.
</div>


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL_ID,
    google_api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)

MOCK_DOC_SOURCE = 'https://docs.google.com/document/d/mock-incident-response-workshop/edit'
MOCK_DOC_TEXT = """
Incident response process in this workshop document:
1. Detect and triage incidents within 15 minutes.
2. Assign incident commander and communication owner.
3. Contain impact first, then analyze root cause.
4. Publish post-incident review with action items within 48 hours.
""".strip()


@tool
def get_workshop_doc() -> str:
    """Return mock workshop document text about incident response."""
    return f"DOCUMENT_TEXT:\n{MOCK_DOC_TEXT}\n\nSOURCE: {MOCK_DOC_SOURCE}"


local_agent = create_react_agent(
    model=llm,
    tools=[get_workshop_doc],
    prompt=(
        'Call get_workshop_doc first. '
        'Answer only from document evidence and include Sources.'
    ),
    checkpointer=InMemorySaver(),
)

print('✅ Локальний mock-агент (ReAct, doc-only) готовий до роботи.')


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🧪 Що відбувається у цій клітинці:</strong><br>
  Викликаємо нашого локального мок-агента тестовим запитом. Він має використати (мок) інструмент і видати структуровану відповідь.
</div>


In [ ]:
def print_local_trajectory(result_dict):
    print("🤖 Локальний агент (Траєкторія):\n" + "-"*60)
    for i, msg in enumerate(result_dict['messages']):
        role = getattr(msg, 'type', type(msg).__name__)

        content = msg.content
        text_blocks = []
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and 'text' in block:
                    text_blocks.append(block['text'])
        elif isinstance(content, str):
            text_blocks.append(content)

        text = "\n".join(text_blocks).strip()
        if text:
            print(f"[{role.upper()}]\n{text}\n")

        if getattr(msg, 'tool_calls', None):
            for tc in msg.tool_calls:
                print(f"🛠️ Виклик інструменту: {tc['name']} з аргументами {tc['args']}\n")

print("🎯 Юзкейс 1: Запит, що базується на документі")
local_result_1 = local_agent.invoke(
    {'messages': [('user', 'What does this document say about incident response?')]},
    config={'configurable': {'thread_id': 'local-react-demo-0001'}},
)
print_local_trajectory(local_result_1)

print("\n" + "="*60 + "\n")
print("🎯 Юзкейс 2: Питання поза документом (очікуємо Not found in document)")
local_result_2 = local_agent.invoke(
    {'messages': [('user', 'What does this document say about SOC 2 controls?')]},
    config={'configurable': {'thread_id': 'local-react-demo-0002'}},
)
print_local_trajectory(local_result_2)


## Step 4 - Deploy runtime through AgentCore CLI

Runtime entrypoint: `runtime_app_agentcore_full.py` (офіційний pattern з `BedrockAgentCoreApp`).


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🚀 Що відбувається у цій клітинці:</strong><br>
  Використовуємо `agentcore` CLI для конфігурації та деплоя нашого коду (`runtime_app_agentcore_full.py`) безпосередньо в рантайм AWS. Цей крок може зайняти певний час (поки статус не стане `READY`).
</div>


In [ ]:
runtime_file = ROOT / 'runtime_app_agentcore_full.py'
if not runtime_file.exists():
    raise FileNotFoundError(runtime_file)

runtime_authorizer = {
    'customJWTAuthorizer': {
        'discoveryUrl': state['inbound']['discovery_url'],
        'allowedClients': [state['inbound']['user_client_id']],
    }
}

config_path = ROOT / '.bedrock_agentcore.yaml'


def clear_stale_runtime_binding(agent_name: str) -> None:
    if not config_path.exists():
        return
    try:
        import yaml
    except Exception:
        return

    data = yaml.safe_load(config_path.read_text()) or {}
    agents = data.get('agents') or {}
    agent_cfg = agents.get(agent_name)
    if not isinstance(agent_cfg, dict):
        return

    runtime_id = ((agent_cfg.get('bedrock_agentcore') or {}).get('agent_id') or '').strip()
    if not runtime_id:
        return

    try:
        ac_control.get_agent_runtime(agentRuntimeId=runtime_id)
        return
    except ac_control.exceptions.ResourceNotFoundException:
        pass
    except Exception:
        return

    agent_cfg.pop('bedrock_agentcore', None)
    config_path.write_text(yaml.safe_dump(data, sort_keys=False))
    print(f'✅ Removed stale runtime binding: {runtime_id}')


def get_bound_runtime_id(agent_name: str) -> str:
    if not config_path.exists():
        return ''
    try:
        import yaml
    except Exception:
        return ''

    data = yaml.safe_load(config_path.read_text()) or {}
    agents = data.get('agents') or {}
    agent_cfg = agents.get(agent_name) or {}
    return str(((agent_cfg.get('bedrock_agentcore') or {}).get('agent_id')) or '').strip()


def list_runtimes_by_name(name: str) -> list[dict[str, Any]]:
    items: list[dict[str, Any]] = []
    next_token = None
    while True:
        req = {'maxResults': 100}
        if next_token:
            req['nextToken'] = next_token
        resp = ac_control.list_agent_runtimes(**req)
        items.extend([x for x in resp.get('agentRuntimes', []) if x.get('agentRuntimeName') == name])
        next_token = resp.get('nextToken')
        if not next_token:
            break
    return items


def wait_runtime_ready_by_id(runtime_id: str, timeout_sec: int = 1200) -> dict[str, Any]:
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        rt = ac_control.get_agent_runtime(agentRuntimeId=runtime_id)
        status = rt.get('status')
        print(f'⏳ Runtime {runtime_id} status: {status}')
        if status == 'READY':
            return rt
        if status in {'CREATE_FAILED', 'UPDATE_FAILED', 'DELETE_FAILED', 'FAILED'}:
            raise RuntimeError(f'Runtime moved to failure status: {status}')
        time.sleep(10)
    raise TimeoutError(f'Runtime {runtime_id} did not reach READY in time')


clear_stale_runtime_binding(names['runtime_agent_name'])

configure_cmd = (
    f"{shlex.quote(AGENTCORE)} configure "
    f"--name {shlex.quote(names['runtime_agent_name'])} "
    f"--entrypoint {shlex.quote(str(runtime_file))} "
    f"--requirements-file {shlex.quote(str(ROOT / 'requirements.txt'))} "
    f"--region {shlex.quote(AWS_REGION)} "
    f"--deployment-type direct_code_deploy "
    f"--runtime PYTHON_3_12 "
    f"--authorizer-config {shlex.quote(json.dumps(runtime_authorizer))} "
    f"--non-interactive"
)
run(configure_cmd)

env_items = {
    'GATEWAY_URL': state['gateway']['gateway_url'],
    'GOOGLE_DOCS_TOOL_NAME': state['gateway']['google_docs_tool_name'],
    'GATEWAY_MCP_VERSION': state['gateway'].get('mcp_version', '2025-11-25'),
    'AWS_REGION': AWS_REGION,
    'GOOGLE_MODEL_ID': GOOGLE_MODEL_ID,
    'GOOGLE_API_KEY': os.environ['GOOGLE_API_KEY'],
}

env_flags = ' '.join(f"--env {shlex.quote(f'{k}={v}')}" for k, v in env_items.items())

deploy_cmd = (
    f"{shlex.quote(AGENTCORE)} deploy "
    f"--agent {shlex.quote(names['runtime_agent_name'])} "
    f"--auto-update-on-conflict "
    f"{env_flags}"
)

deploy_out = run(deploy_cmd, check=False)
if run.last_rc != 0 and 'ResourceNotFoundException' in deploy_out and 'UpdateAgentRuntime' in deploy_out:
    print('⚠️ CLI tried to update deleted runtime, retrying once after cleanup...')
    clear_stale_runtime_binding(names['runtime_agent_name'])
    run(configure_cmd)
    deploy_out = run(deploy_cmd, check=False)

if run.last_rc != 0 and 'ConflictException' in deploy_out and "while it's CREATING" in deploy_out:
    print('⚠️ Runtime is still CREATING. Waiting 30s, then retrying once...')
    time.sleep(30)
    deploy_out = run(deploy_cmd, check=False)

if run.last_rc != 0:
    raise RuntimeError('Deploy failed. Stop here and fix deploy before invoke.')

runtime_id = get_bound_runtime_id(names['runtime_agent_name'])
if not runtime_id:
    candidates = sorted(
        list_runtimes_by_name(names['runtime_agent_name']),
        key=lambda x: str(x.get('lastUpdatedAt', '')),
        reverse=True,
    )
    if not candidates:
        raise RuntimeError('Deploy succeeded but no runtime found by name.')
    runtime_id = candidates[0]['agentRuntimeId']

rt = wait_runtime_ready_by_id(runtime_id)
state['runtime'] = {
    'runtime_id': rt['agentRuntimeId'],
    'runtime_arn': rt['agentRuntimeArn'],
    'runtime_status': rt.get('status'),
}
print('✅ Runtime ready:', json.dumps(state['runtime'], indent=2))


## Step 5 - Runtime invoke + consent

1. Перший invoke повертає consent URL (якщо Google OAuth ще не прив'язаний до сесії).
2. Після consent запускаєш другий invoke з тим самим `oauth_session_uri`.


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>📞 Що відбувається у цій клітинці:</strong><br>
  Робимо **перший реальний запит (invoke)** до нашого розгорнутого у хмарі Runtime агента. Якщо ми ще не надавали дозвіл для Google (OAuth), то агент зупиниться і поверне посилання на отримання згоди (`authorization_url`).
</div>


> Якщо consent ще не виданий, у відповіді буде `authorization_url` та `oauth_session_uri`.

In [ ]:
runtime_thread_id = os.getenv('RUNTIME_THREAD_ID', 'm11-runtime-react-demo-000000000000001')
if len(runtime_thread_id) < 33:
    raise ValueError('RUNTIME_THREAD_ID must be >= 33 chars for AgentCore runtime session header.')

mcp_session_id = runtime_thread_id

if 'runtime' not in state or not state['runtime'].get('runtime_arn'):
    runtime_id = ''
    if 'get_bound_runtime_id' in globals():
        runtime_id = get_bound_runtime_id(names['runtime_agent_name'])

    if runtime_id:
        rt = ac_control.get_agent_runtime(agentRuntimeId=runtime_id)
    else:
        items = []
        next_token = None
        while True:
            req = {'maxResults': 100}
            if next_token:
                req['nextToken'] = next_token
            resp = ac_control.list_agent_runtimes(**req)
            items.extend([x for x in resp.get('agentRuntimes', []) if x.get('agentRuntimeName') == names['runtime_agent_name']])
            next_token = resp.get('nextToken')
            if not next_token:
                break
        items = sorted(items, key=lambda x: str(x.get('lastUpdatedAt', '')), reverse=True)
        if not items:
            raise RuntimeError('No runtime found. Run Step 4 deploy first.')
        rt = ac_control.get_agent_runtime(agentRuntimeId=items[0]['agentRuntimeId'])

    state['runtime'] = {
        'runtime_id': rt['agentRuntimeId'],
        'runtime_arn': rt['agentRuntimeArn'],
        'runtime_status': rt.get('status'),
    }

runtime_arn = state['runtime']['runtime_arn']
runtime_url = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{urllib.parse.quote(runtime_arn, safe='')}/invocations"


def detect_expected_app_version() -> str:
    env_version = os.getenv('EXPECTED_APP_VERSION', '').strip()
    if env_version:
        return env_version

    runtime_src = ROOT / 'runtime_app_agentcore_full.py'
    if runtime_src.exists():
        m = re.search(r"^APP_VERSION\s*=\s*['\"]([^'\"]+)['\"]", runtime_src.read_text(), flags=re.MULTILINE)
        if m:
            return m.group(1)

    return ''


def print_runtime_result(label: str, data: dict):
    print(f'\n=== {label} ===')
    print('app_version:', data.get('app_version'))
    print('recursion_limit:', data.get('recursion_limit'))
    print('consent_required:', data.get('consent_required'))
    print('tools_used:', data.get('tools_used'))
    print('tool_call_counts:', data.get('tool_call_counts'))
    print('tool_call_limits:', data.get('tool_call_limits'))
    print('response:\n', data.get('response', ''))
    trace = data.get('tool_trace') or []
    if trace:
        print('\nTool trace:')
        for row in trace:
            print(f"- step={row.get('step')} event={row.get('event')} tool={row.get('tool')}")


payload1 = {
    'prompt': os.getenv('RUNTIME_PROMPT_1', 'Summarize incident response from this document in 6 bullets and include source.'),
    'doc_id': os.environ['GOOGLE_DOC_ID'],
    'user_access_token': user_access_token,
    'thread_id': runtime_thread_id,
    'mcp_session_id': mcp_session_id,
    'max_steps': 5,
    'max_doc_calls': int(os.getenv('RUNTIME_MAX_DOC_CALLS', '1')),
}

resp1 = requests.post(
    runtime_url,
    params={'qualifier': 'DEFAULT'},
    headers={
        'Authorization': f'Bearer {user_access_token}',
        'Content-Type': 'application/json',
        'Accept': 'application/json',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': runtime_thread_id,
    },
    json=payload1,
    timeout=(20, 90),
)

print('HTTP status:', resp1.status_code)
if not resp1.ok:
    raise RuntimeError(f'Runtime invoke failed: {resp1.status_code} {resp1.text[:500]}')

runtime_first_result = resp1.json()
print_runtime_result('FIRST INVOKE', runtime_first_result)

expected_app_version = detect_expected_app_version()
actual_app_version = str(runtime_first_result.get('app_version') or '')
print('expected_app_version:', expected_app_version or '<not-set>')
if expected_app_version and actual_app_version != expected_app_version:
    raise RuntimeError(
        f"Stale runtime detected: app_version={actual_app_version!r}, expected={expected_app_version!r}. Re-run Step 4 deploy, then run this cell again."
    )

authorization_url = (runtime_first_result.get('authorization_url') or '').strip()
oauth_session_uri = (runtime_first_result.get('oauth_session_uri') or '').strip()
state['runtime_oauth_session_uri'] = oauth_session_uri

if authorization_url:
    print('')
    print('Open authorization_url in browser, complete consent, then run next cell.')


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔄 Що відбувається у цій клітинці:</strong><br>
  Після надання згоди (OAuth consent) ми здійснюємо **другий запит**, щоб повідомити рантайму, що токен готовий, і агент може продовжити обробку запиту. Буде виведено фінальну, змістовну відповідь RAG-агента.
</div>


> Після Google consent браузер може перейти на `http://localhost:8081/...` і показати сторінку помилки з'єднання. Для цього воркшопу це нормально: повернись у ноутбук і виконай наступну клітинку.

In [ ]:
oauth_session_uri = (
    (runtime_first_result.get('oauth_session_uri') if 'runtime_first_result' in globals() else '')
    or state.get('runtime_oauth_session_uri', '')
    or os.getenv('RUNTIME_OAUTH_SESSION_URI', '')
).strip()

if oauth_session_uri:
    ac_data = session.client('bedrock-agentcore', region_name=AWS_REGION)
    try:
        ac_data.complete_resource_token_auth(
            userIdentifier={'userToken': user_access_token},
            sessionUri=oauth_session_uri,
        )
        print('OAuth session completed in AgentCore token vault.')
    except ClientError as exc:
        code = exc.response.get('Error', {}).get('Code', 'Unknown')
        msg = exc.response.get('Error', {}).get('Message', str(exc))
        raise RuntimeError(f'complete_resource_token_auth failed: {code} - {msg}')
else:
    print('No oauth_session_uri: consent already granted (or not required).')

skip_second_invoke = os.getenv('SKIP_SECOND_INVOKE', '0').strip().lower() in {'1', 'true', 'yes'}

if skip_second_invoke:
    print('')
    print('Second invoke is skipped because SKIP_SECOND_INVOKE=1.')
    print_runtime_result('FIRST INVOKE (CURRENT)', runtime_first_result)
else:
    payload2 = {
        'prompt': os.getenv(
            'RUNTIME_PROMPT_2',
            'Answer in 6 bullets: summarize incident response from the document and cite source link from the document.',
        ),
        'doc_id': os.environ['GOOGLE_DOC_ID'],
        'user_access_token': user_access_token,
        'thread_id': runtime_thread_id,
        'mcp_session_id': mcp_session_id,
        'max_steps': 5,
        'max_doc_calls': int(os.getenv('RUNTIME_MAX_DOC_CALLS', '1')),
    }

    def invoke_runtime_with_retry(payload: dict, max_attempts: int = 2, read_timeout_sec: int = 90):
        last_exc = None
        for attempt in range(1, max_attempts + 1):
            try:
                return requests.post(
                    runtime_url,
                    params={'qualifier': 'DEFAULT'},
                    headers={
                        'Authorization': f'Bearer {user_access_token}',
                        'Content-Type': 'application/json',
                        'Accept': 'application/json',
                        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': runtime_thread_id,
                    },
                    json=payload,
                    timeout=(20, read_timeout_sec),
                )
            except requests.exceptions.ReadTimeout as exc:
                last_exc = exc
                print(f'Runtime invoke timeout on attempt {attempt}/{max_attempts}. Retrying...')
                time.sleep(3)
        raise last_exc

    resp2 = invoke_runtime_with_retry(payload2)

    print('HTTP status:', resp2.status_code)
    if not resp2.ok:
        raise RuntimeError(f'Runtime invoke #2 failed: {resp2.status_code} {resp2.text[:500]}')

    runtime_second_result = resp2.json()
    print_runtime_result('SECOND INVOKE', runtime_second_result)

    expected_app_version = detect_expected_app_version()
    actual_app_version = str(runtime_second_result.get('app_version') or '')
    print('expected_app_version:', expected_app_version or '<not-set>')
    if expected_app_version and actual_app_version != expected_app_version:
        raise RuntimeError(
            f"Stale runtime detected on second invoke: app_version={actual_app_version!r}, expected={expected_app_version!r}."
        )


## Step 6 - Сleanup

Корисні команди:
- `agentcore status --agent <name>`
- `agentcore obs --help`
- `agentcore destroy --agent <name>`
- `agentcore identity cleanup --region <region>`


<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🗑️ Що відбувається у цій клітинці:</strong><br>
  Ця остання клітинка, клітинка номер 6, призначена для **повного очищення створених ресурсів** 🧹. Тут зібрано Python-функцію `cleanup_all()`, яка послідовно проходить по всьому, що ми створили у цьому воркшопі (Agent Runtime, API Gateway, Targets, OAuth Provider та Cognito User Pool), і акуратно їх видаляє. Це допоможе уникнути зайвих витрат в AWS. Щоб запустити очищення, потрібно просто розкоментувати і викликати `cleanup_all()` у кінці клітинки.
</div>


In [ ]:
def cleanup_all():
    print("Starting cleanup of workshop resources...\n")

    # 1. Delete Agent Runtime via CLI
    agent_name = names.get('runtime_agent_name')
    if agent_name:
        print(f"Deleting Agent Runtime ({agent_name}) via CLI...")
        destroy_cmd = f"{shlex.quote(AGENTCORE)} destroy --agent {shlex.quote(agent_name)} --force"
        run(destroy_cmd, check=False)
        print("Agent Runtime delete command finished.\n")

    # 2. Delete Gateway Targets for active gateway
    gateway_id = state.get('gateway', {}).get('gateway_id')
    if gateway_id:
        print(f"Looking up Gateway Targets for ({gateway_id})...")
        try:
            targets_resp = ac_control.list_gateway_targets(gatewayIdentifier=gateway_id)
            targets = targets_resp.get('items', [])
            for t in targets:
                t_id = t['targetId']
                print(f"Deleting Gateway Target ({t_id})...")
                ac_control.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=t_id)
                print(f"Gateway Target {t_id} deleted.")
        except Exception as e:
            print(f"Warning: failed deleting Gateway Targets: {e}\n")

    # 3. Delete all gateways with workshop prefix (including stale bootstrap)
    try:
        items = []
        next_token = None
        while True:
            req = {'maxResults': 100}
            if next_token:
                req['nextToken'] = next_token
            resp = ac_control.list_gateways(**req)
            items.extend(resp.get('items', []))
            next_token = resp.get('nextToken')
            if not next_token:
                break

        for g in items:
            g_name = g.get('name', '')
            if g_name.startswith(names.get('gateway_name', '')):
                g_id = g['gatewayId']
                print(f"Deleting Gateway ({g_name}/{g_id})...")
                try:
                    ac_control.delete_gateway(gatewayIdentifier=g_id)
                    print("Gateway deleted.")
                except Exception as e:
                    print(f"Warning: {e}")
        print('')
    except Exception as e:
        print(f"Warning while listing/deleting gateways: {e}\n")

    # 4. Delete OAuth Provider
    provider_name = names.get('provider_name')
    if provider_name:
        print(f"Deleting OAuth Provider ({provider_name})...")
        try:
            ac_control.delete_oauth2_credential_provider(name=provider_name)
            print("OAuth Provider deleted.\n")
        except Exception as e:
            print(f"Warning: {e}\n")


    # 5. Delete Cognito User Pool
    pool_id = state.get('inbound', {}).get('user_pool_id')
    if pool_id:
        print(f"Deleting Cognito User Pool ({pool_id})...")
        try:
            cognito.delete_user_pool(UserPoolId=pool_id)
            print("Cognito User Pool deleted.\n")
        except Exception as e:
            print(f"Warning: {e}\n")

    print("Cleanup completed.")


# Uncomment and run when you really want to delete resources:
# cleanup_all()
